**Task 0 (setup):**

Recreate the exact classification preprocessing from Part 2 (same label encoding for `cut`, same one-hot encoding for `color`/`clarity`, same `train_test_split(random_state=42)`, same `StandardScaler` fit only on train) so that `X_train_clf_scaled`, `X_test_clf_scaled`, `y_clf_train`, `y_clf_test` here are identical to Part 2's.

In [1]:
# import essential libraries
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
# load cleaned dataset from Part 1
df = pd.read_csv("cleaned_data.csv")

# recreate classification label exactly as Part 2
y_clf = (df["price"] > df["price"].median()).astype(int)
X = df.drop(columns=["price"])

# recreate cut label-encoding exactly as Part 2
cut_order = {"Fair": 0, "Good": 1, "Very Good": 2, "Premium": 3, "Ideal": 4}
X["cut"] = X["cut"].map(cut_order)

# recreate color/clarity one-hot encoding exactly as Part 2
X = pd.get_dummies(X, columns=["color", "clarity"], drop_first=True)

# recreate the same train/test split (same random_state=42)
X_train_clf, X_test_clf, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

# recreate scaling (fit only on train)
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

feature_names = X_train_clf.columns.tolist()
print("X_train_clf_scaled:", X_train_clf_scaled.shape)
print("X_test_clf_scaled :", X_test_clf_scaled.shape)

X_train_clf_scaled: (43035, 20)
X_test_clf_scaled : (10759, 20)


**Task 1:**

Decision Tree baseline: train a `DecisionTreeClassifier` with no hyperparameter constraints (`max_depth=None`) on `X_train_clf_scaled`/`y_clf_train`. Report training accuracy and test accuracy.

In [3]:
# unconstrained decision tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train_clf_scaled, y_clf_train)

train_acc_full = accuracy_score(y_clf_train, dt_full.predict(X_train_clf_scaled))
test_acc_full = accuracy_score(y_clf_test, dt_full.predict(X_test_clf_scaled))

print("Train accuracy:", train_acc_full)
print("Test accuracy :", test_acc_full)
print("Train-test gap:", train_acc_full - test_acc_full)

Train accuracy: 0.9999302892994074
Test accuracy : 0.9661678594664932
Train-test gap: 0.03376242983291422


**Task 2:**

Controlled Decision Tree: train a second `DecisionTreeClassifier` with `max_depth=5` and `min_samples_split=20`. Report training and test accuracy again.

In [4]:
# controlled decision tree
dt_ctrl = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_ctrl.fit(X_train_clf_scaled, y_clf_train)

train_acc_ctrl = accuracy_score(y_clf_train, dt_ctrl.predict(X_train_clf_scaled))
test_acc_ctrl = accuracy_score(y_clf_test, dt_ctrl.predict(X_test_clf_scaled))

print("Train accuracy:", train_acc_ctrl)
print("Test accuracy :", test_acc_ctrl)
print("Train-test gap:", train_acc_ctrl - test_acc_ctrl)

Train accuracy: 0.9637271987916812
Test accuracy : 0.9640301143228924
Train-test gap: -0.00030291553121120085


**Task 3:**

Gini vs Entropy comparison: train two `DecisionTreeClassifier` models with `max_depth=5`, one with `criterion='gini'` and one with `criterion='entropy'`. Report the test accuracy of each.

In [5]:
# gini vs entropy
dt_gini = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=42)
dt_gini.fit(X_train_clf_scaled, y_clf_train)

dt_ent = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=42)
dt_ent.fit(X_train_clf_scaled, y_clf_train)

acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_clf_scaled))
acc_ent = accuracy_score(y_clf_test, dt_ent.predict(X_test_clf_scaled))

print("Gini test accuracy   :", acc_gini)
print("Entropy test accuracy:", acc_ent)

Gini test accuracy   : 0.9640301143228924
Entropy test accuracy: 0.9628218235895529


**Task 4:**

Random Forest: train a `RandomForestClassifier` with `n_estimators=100`, `max_depth=10`, `random_state=42`. Report training accuracy, test accuracy, and ROC-AUC. Retrieve `feature_importances_` and display the top 5 features.

In [6]:
# random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_clf_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf.predict(X_train_clf_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf.predict(X_test_clf_scaled))
rf_auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test_clf_scaled)[:, 1])

print("Train accuracy:", rf_train_acc)
print("Test accuracy :", rf_test_acc)
print("Test AUC      :", rf_auc)

Train accuracy: 0.9799697920297432
Test accuracy : 0.9736964401896087
Test AUC      : 0.9973331448833586


In [7]:
# top 5 features by importance
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("Top 5 features by importance:")
print(importances.head(5))

Top 5 features by importance:
y              0.363059
carat          0.211711
x              0.180786
z              0.179954
clarity_SI2    0.014069
dtype: float64


**Task 4a:**

Gradient Boosting: train a `GradientBoostingClassifier` with `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `random_state=42`. Report training accuracy, test accuracy, and ROC-AUC.

In [8]:
# gradient boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_clf_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb.predict(X_train_clf_scaled))
gb_test_acc = accuracy_score(y_clf_test, gb.predict(X_test_clf_scaled))
gb_auc = roc_auc_score(y_clf_test, gb.predict_proba(X_test_clf_scaled)[:, 1])

print("Train accuracy:", gb_train_acc)
print("Test accuracy :", gb_test_acc)
print("Test AUC      :", gb_auc)

Train accuracy: 0.9752294643894505
Test accuracy : 0.9737893856306348
Test AUC      : 0.997378431623933


**Task 4b:**

Feature ablation study: identify the 5 lowest-importance features from the Random Forest in Task 4, remove them, retrain an identical Random Forest, and compare test-set AUC with vs without them.

In [9]:
# feature ablation study
lowest5 = importances.tail(5).index.tolist()
print("5 lowest-importance features:", lowest5)

keep_idx = [i for i, f in enumerate(feature_names) if f not in lowest5]
X_train_reduced = X_train_clf_scaled[:, keep_idx]
X_test_reduced = X_test_clf_scaled[:, keep_idx]

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = rf_auc
auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print("Full-model AUC   :", auc_full)
print("Reduced-model AUC:", auc_reduced)

5 lowest-importance features: ['clarity_IF', 'clarity_VS1', 'clarity_VS2', 'color_G', 'color_F']


Full-model AUC   : 0.9973331448833586
Reduced-model AUC: 0.9967326160945725


**Task 5:**

Cross-validated comparison: use `cross_val_score` with `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` and `scoring='roc_auc'` to evaluate Logistic Regression, the controlled Decision Tree, Random Forest, and Gradient Boosting. Report mean and std of the 5-fold AUC for each.

In [10]:
# cross-validated comparison
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree(depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}

cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X_train_clf_scaled, y_clf_train, cv=skf, scoring="roc_auc")
    cv_results[name] = (scores.mean(), scores.std())
    print(f"{name}: mean AUC={scores.mean():.4f}  std={scores.std():.4f}")

LogisticRegression: mean AUC=0.9982  std=0.0002


DecisionTree(depth=5): mean AUC=0.9928  std=0.0005


RandomForest: mean AUC=0.9972  std=0.0002


GradientBoosting: mean AUC=0.9969  std=0.0003


**Task 6:**

Hyperparameter tuning with GridSearchCV on a Random Forest pipeline (`SimpleImputer` -> `StandardScaler` -> `RandomForestClassifier`). Grid: `n_estimators` in [50,100,200], `max_depth` in [5,10,None], `min_samples_leaf` in [1,5], with `StratifiedKFold(5)` and `scoring='roc_auc'`.

In [11]:
# gridsearchcv pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5],
}

grid = GridSearchCV(pipeline, param_grid, cv=skf, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train_clf, y_clf_train)  # unscaled -- pipeline handles scaling

print("Best params  :", grid.best_params_)
print("Best CV score:", grid.best_score_)

n_configs = 1
for v in param_grid.values():
    n_configs *= len(v)
print(f"Total configurations evaluated: {n_configs} x 5 folds = {n_configs * 5}")

Best params  : {'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}
Best CV score: 0.9977599898397713
Total configurations evaluated: 18 x 5 folds = 90


In [12]:
# evaluate best pipeline on the held-out test set
best_pipeline = grid.best_estimator_
test_auc_best = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test_clf)[:, 1])
print("Best pipeline test AUC:", test_auc_best)

Best pipeline test AUC: 0.9979553235332436


**Task 7:**

Manual learning curve: using the best pipeline from Task 6, train on 20%, 40%, 60%, 80%, and 100% of `X_train_clf`. For each fraction compute training AUC (on that same subset) and test AUC (on the fixed, full test set). Print as a Threshold-style table: Training fraction | Training AUC | Test AUC.

In [13]:
# manual learning curve
lc_rows = []
best_params_clean = {k.split("__")[1]: v for k, v in grid.best_params_.items()}

for f in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_rows = int(f * len(X_train_clf))
    X_sub = X_train_clf.iloc[:n_rows]
    y_sub = y_clf_train.iloc[:n_rows]

    pipe_f = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        RandomForestClassifier(random_state=42, **best_params_clean)
    )
    pipe_f.fit(X_sub, y_sub)

    train_auc = roc_auc_score(y_sub, pipe_f.predict_proba(X_sub)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipe_f.predict_proba(X_test_clf)[:, 1])

    lc_rows.append({"Training fraction": f, "Training AUC": train_auc, "Test AUC": test_auc})

lc_table = pd.DataFrame(lc_rows)
print(lc_table)

   Training fraction  Training AUC  Test AUC
0                0.2           1.0  0.996662
1                0.4           1.0  0.997499
2                0.6           1.0  0.997674
3                0.8           1.0  0.997737
4                1.0           1.0  0.997955


**Task 8:**

Serialize the best model: save the best pipeline with `joblib.dump`, then reload it with `joblib.load` and call `.predict()` on two hand-crafted test rows to confirm it runs without errors.

In [14]:
# serialize best pipeline (compressed -- GitHub's web uploader caps drag-and-drop files at 25MB;
# compress=6 shrinks this from ~42MB to ~7MB with identical predictions)
joblib.dump(best_pipeline, "best_model.pkl", compress=6)
print("Saved best_model.pkl (compressed)")

Saved best_model.pkl (compressed)


In [15]:
# reload-and-predict demo
loaded = joblib.load("best_model.pkl")

sample_rows = X_test_clf.iloc[:2]
preds = loaded.predict(sample_rows)
probs = loaded.predict_proba(sample_rows)

print(sample_rows)
print("Predictions  :", preds)
print("Probabilities:", probs)

       carat  cut  depth  table     x     y     z  color_E  color_F  color_G  \
43520   0.71    0   64.9   54.0  5.63  5.53  3.62    False    False    False   
4264    0.90    2   61.0   59.0  6.14  6.18  3.76     True    False    False   

       color_H  color_I  color_J  clarity_IF  clarity_SI1  clarity_SI2  \
43520    False    False     True       False        False        False   
4264     False    False    False       False        False         True   

       clarity_VS1  clarity_VS2  clarity_VVS1  clarity_VVS2  
43520        False         True         False         False  
4264         False        False         False         False  
Predictions  : [0 1]
Probabilities: [[0.755 0.245]
 [0.    1.   ]]


**Summary comparison table:**

Compare all models from Parts 2 and 3 on 5-fold CV mean AUC, 5-fold CV std AUC, and test-set AUC.

In [16]:
# summary comparison table
summary_rows = []
for name, (mean_auc, std_auc) in cv_results.items():
    summary_rows.append({"Model": name, "CV Mean AUC": mean_auc, "CV Std AUC": std_auc})

summary_rows.append({
    "Model": "RandomForest (GridSearch best)",
    "CV Mean AUC": grid.best_score_,
    "CV Std AUC": np.nan
})

summary_df = pd.DataFrame(summary_rows)

# refit logistic regression once more to get its test AUC for the table
logreg_refit = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_clf_scaled, y_clf_train)

test_aucs = {
    "LogisticRegression": roc_auc_score(y_clf_test, logreg_refit.predict_proba(X_test_clf_scaled)[:, 1]),
    "DecisionTree(depth=5)": roc_auc_score(y_clf_test, dt_ctrl.predict_proba(X_test_clf_scaled)[:, 1]),
    "RandomForest": rf_auc,
    "GradientBoosting": gb_auc,
    "RandomForest (GridSearch best)": test_auc_best,
}

summary_df["Test AUC"] = summary_df["Model"].map(test_aucs)
print(summary_df)

                            Model  CV Mean AUC  CV Std AUC  Test AUC
0              LogisticRegression     0.998219    0.000170  0.997926
1           DecisionTree(depth=5)     0.992767    0.000549  0.993090
2                    RandomForest     0.997199    0.000176  0.997333
3                GradientBoosting     0.996949    0.000318  0.997378
4  RandomForest (GridSearch best)     0.997760         NaN  0.997955
